In [ ]:
# Cell 1 — Imports
import numpy as np
import matplotlib.pyplot as plt
import trackpy as tp

from multiprocessing import Queue
from pathlib import Path
import cv2

from Training_data_generation.Image_creation.Image_generator import generate_image
from Training_data_generation.Yolo_export import worker_loop
from Training_data_generation.Image_creation.Triangle_Sampler_Class import TrianglePrismSampler

In [ ]:
# Cell 2 — Helper: overlay outlines on image
def show_overlay(img_u8, outlines_px, title="overlay"):
    plt.figure(figsize=(6,6))
    plt.imshow(img_u8, cmap="gray")

    for poly in outlines_px:
        if poly is None or len(poly) < 3:
            continue
        poly = np.asarray(poly)

        # close polygon
        x = np.r_[poly[:,0], poly[0,0]]
        y = np.r_[poly[:,1], poly[0,1]]
        plt.plot(x, y, linewidth=1)

    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# Cell 3 — Generate exactly 1 sample using worker_loop (queue)
q = Queue(maxsize=1)
worker_loop(q, n=1)
img_u8, lines = q.get()

print("image shape:", img_u8.shape)
print("num label lines:", len(lines))

plt.figure(figsize=(6,6))
plt.imshow(img_u8, cmap="gray")
plt.axis("off")
plt.show()

print("Example label line:", lines[0] if lines else "NO LABELS")

In [ ]:
# Cell 4 — Manual sampler + DeepTrack image generation (if you want direct access to outlines)
sampler = TrianglePrismSampler(
    H=2048,
    W=2048,
    margin=50,
    side_length_px=30,
    thickness_px=8,
)

image_of_particles = generate_image(
    sampler=sampler,
    pos_sampler=sampler.pos_sampler,
    rot_sampler=sampler.rot_sampler,
)

sampler.reset()

img = image_of_particles.update(verbose=True).resolve()
img_u8_manual = (img * 255).astype(np.uint8)

outlines_px = sampler.outlines_px
show_overlay(img_u8_manual, outlines_px, title="manual sampler overlay")

